# Win outcomes scout — inventory queries

Coverage statistics for `analytics/win_outcomes_scout/INVENTORY.md`. Each cell maps to an `[NB-N]` reference in the inventory doc. Run on Databricks; lift the resulting numbers back into the doc.

**Execution:** Databricks Workspace (preferred) or via Databricks Connect from a local kernel. See `analytics/win_churn/WORKFLOW.md` for the sync pattern.

**Memory reminder:** the project memory says `for ad-hoc queries: use prod path` — hit `goodparty_data_catalog.*` directly. `ref()` can resolve to stale dev artifacts.

## Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Configured automatically inside a Databricks notebook; the SparkSession.builder line
# is for Databricks Connect execution from a local kernel.
try:
    spark  # type: ignore[name-defined]
except NameError:
    spark = SparkSession.builder.getOrCreate()

CATALOG = "goodparty_data_catalog"
ANALYTICS = f"{CATALOG}.mart_analytics"
CIVICS = f"{CATALOG}.mart_civics"
INTERMEDIATE = f"{CATALOG}.dbt_intermediate"  # adjust if the intermediate schema is named differently in your workspace

def show_count(df, label):
    n = df.count()
    print(f"{label}: {n:,}")
    return n

## NB-1 — Volume sanity on `users_win_candidacy`

Maps to INVENTORY.md "Source 1.2 / Observed coverage". Establishes the row count, distinct `user_id`, and distinct `campaign_id`.

In [ ]:
uwc = spark.table(f"{ANALYTICS}.users_win_candidacy").filter(~F.col("is_demo"))
show_count(uwc, "users_win_candidacy rows (non-demo)")
show_count(uwc.select("user_id").distinct(), "distinct user_id")
show_count(uwc.select("campaign_id").distinct(), "distinct campaign_id")
show_count(uwc.filter(F.col("is_latest_version")), "is_latest_version = true")

## NB-2 — Outcome and viability availability

Maps to INVENTORY.md "Source 1.2". What fraction of rows have a captured stage outcome / general result / viability score?

In [ ]:
any_outcome = F.coalesce(
    F.col("primary_election_result"),
    F.col("general_election_result"),
    F.col("primary_runoff_election_result"),
    F.col("general_runoff_election_result"),
)
uwc_latest = uwc.filter(F.col("is_latest_version"))
uwc_latest.agg(
    F.count("*").alias("rows"),
    F.count(any_outcome).alias("any_outcome_populated"),
    F.count("general_election_result").alias("general_election_result_populated"),
    F.count("viability_score").alias("viability_score_populated"),
    F.count("win_number").alias("win_number_populated"),
).show(truncate=False)

## NB-3 — Spot-check: candidacy_result fallback hazard

The doc warns that `candidacy_result` can read 'Won' when only the primary was won. Quantify the cells where `candidacy_result = 'Won'` but `general_election_result` is null or not 'Won'.

In [ ]:
spark.sql(f"""
select
    coalesce(candidacy_result, 'NULL') as candidacy_result,
    coalesce(general_election_result, 'NULL') as general_election_result,
    count(*) as rows
from {CIVICS}.candidacy
group by 1, 2
order by 1, 2
""").show(100, truncate=False)

## NB-4 — Domain values for segmentation columns

Maps to INVENTORY.md "Source 1.4". Enumerate the actual values seen for `election_level`, `office_type`, `campaign_state`, and `campaign_party`.

In [ ]:
for col in ("election_level", "office_type", "campaign_party"):
    print(f"=== {col} ===")
    (
        uwc_latest.groupBy(col).count().orderBy(F.desc("count")).show(30, truncate=False)
    )

print("=== campaign_state (top 20) ===")
uwc_latest.groupBy("campaign_state").count().orderBy(F.desc("count")).show(20)

## NB-5 — 2025 archive vs 2026+ shape mismatch

Maps to INVENTORY.md "Source 2.2". Slice candidacies by cycle and source_systems combination.

In [ ]:
spark.sql(f"""
with c as (
    select
        year(general_election_date) as election_year,
        sort_array(source_systems) as sources,
        viability_score,
        is_incumbent,
        is_open_seat,
        candidacy_result
    from {CIVICS}.candidacy
)
select
    election_year,
    sources,
    count(*) as rows,
    count(viability_score) as viability_populated,
    count(is_incumbent) as incumbent_populated,
    count(is_open_seat) as open_seat_populated,
    count(candidacy_result) as candidacy_result_populated
from c
group by 1, 2
order by 1, 3 desc
""").show(60, truncate=False)

## NB-6 — Viability score coverage by slice

Maps to INVENTORY.md "Source 4.3". Sliced by office level, state, cycle, and Pro.

In [ ]:
# By election_level
uwc_latest.groupBy("election_level").agg(
    F.count("*").alias("rows"),
    F.count("viability_score").alias("viability_populated"),
    (F.count("viability_score") / F.count("*")).alias("coverage_pct"),
).orderBy(F.desc("rows")).show(20, truncate=False)

# By is_pro
uwc_latest.groupBy("is_pro").agg(
    F.count("*").alias("rows"),
    F.count("viability_score").alias("viability_populated"),
    (F.count("viability_score") / F.count("*")).alias("coverage_pct"),
).show(truncate=False)

# By election cycle
uwc_latest.withColumn("cycle", F.year("general_election_date")).groupBy("cycle").agg(
    F.count("*").alias("rows"),
    F.count("viability_score").alias("viability_populated"),
    (F.count("viability_score") / F.count("*")).alias("coverage_pct"),
).orderBy("cycle").show(20, truncate=False)

# By state (top 20 by row count)
uwc_latest.groupBy("campaign_state").agg(
    F.count("*").alias("rows"),
    F.count("viability_score").alias("viability_populated"),
    (F.count("viability_score") / F.count("*")).alias("coverage_pct"),
).orderBy(F.desc("rows")).show(20, truncate=False)

## NB-7 — Viability score calibration on closed races

Maps to INVENTORY.md "Source 4.6". Observed win rate by viability score decile, restricted to candidacies with both a non-null `viability_score` and a `general_election_result` in (`Won`, `Lost`).

**Caveat from the doc:** `viability_score` is overwritten on each BR refresh, so calibration may be contaminated if BR refreshed after results came in. Treat as an upper bound on calibration quality.

In [ ]:
calibrate = (
    uwc_latest
    .filter(F.col("viability_score").isNotNull())
    .filter(F.col("general_election_result").isin("Won", "Lost"))
    .select("viability_score", "general_election_result")
)
show_count(calibrate, "calibration-eligible candidacies")

# Decile buckets via ntile
from pyspark.sql.window import Window
w = Window.orderBy("viability_score")
calibrated = calibrate.withColumn("decile", F.ntile(10).over(w))
calibrated.groupBy("decile").agg(
    F.count("*").alias("n"),
    F.min("viability_score").alias("min_score"),
    F.max("viability_score").alias("max_score"),
    F.avg(F.when(F.col("general_election_result") == "Won", 1.0).otherwise(0.0)).alias("win_rate"),
).orderBy("decile").show(15, truncate=False)

## NB-8 — Recommended-action funnel: does product DB carry structured task data?

Maps to INVENTORY.md "Source 3.8" and "Source 5.4". Inspect `path_to_victory` and any adjacent product tables for a task / step / recommended-action schema.

In [ ]:
# Schema introspection on path_to_victory + adjacent product tables.
# Adjust schema name if your dbt staging schema is different.
for tbl in (
    "stg_airbyte_source__gp_api_db_path_to_victory",
    "stg_airbyte_source__gp_api_db_outreach",
    "stg_airbyte_source__gp_api_db_campaign",
):
    print(f"=== {tbl} ===")
    try:
        df = spark.table(f"{CATALOG}.dbt_staging.{tbl}")
        df.printSchema()
        df.limit(3).show(truncate=80)
    except Exception as e:
        print(f"(could not load: {e})")
    print()

## NB-9 — Amplitude event-type inventory (the broader "what else exists?" question)

The dbt intermediate models filter to specific event_types. This cell scans the staging events table to find Amplitude event types that exist but are NOT currently aggregated — they're candidates for instrumentation-as-already-tracked-but-not-surfaced.

In [ ]:
# WARNING: scanning all Amplitude events is expensive. Restrict to a recent window.
spark.sql(f"""
select event_type, count(*) as events, count(distinct user_id) as users
from {CATALOG}.dbt_staging.stg_airbyte_source__amplitude_api_events
where event_time >= current_date - interval 30 days
group by event_type
order by events desc
limit 100
""").show(100, truncate=False)

## NB-10 — Activity per user during their cycle: join engagement to outcome

Sanity check that the keystone join works. Pull a small sample of candidacies that have both an outcome and engagement to confirm the join is sound.

In [ ]:
spark.sql(f"""
with uwc as (
    select user_id, campaign_id, campaign_state, election_level, office_type,
           is_pro, viability_score, general_election_date, general_election_result
    from {ANALYTICS}.users_win_candidacy
    where is_latest_version and not is_demo
      and general_election_result in ('Won', 'Lost')
),
engagement as (
    select user_id,
           sum(total_events) as total_events,
           sum(activity_days) as activity_days,
           sum(campaigns_sent) as campaigns_sent,
           sum(dashboard_views) as dashboard_views
    from {INTERMEDIATE}.int__amplitude_win_activity_weekly
    group by user_id
)
select uwc.*, e.total_events, e.activity_days, e.campaigns_sent, e.dashboard_views
from uwc
left join engagement e using (user_id)
order by uwc.general_election_date desc
limit 50
""").show(truncate=False)

---

## Wrap-up

Lift the key numbers from these cells into the `[NB-N]` placeholders in `../INVENTORY.md`. After the lift, walk through the inventory once with real numbers to confirm the narrative still holds, then mark the inventory "verified" in `SESSION_NOTES.md`.